# Modelo de Traslado de Valores - FINABIEN

## 0.4 Necesidades de Efectivo

### El objetivo de este 'notebook' es hacer las proyecciones a futuro del flujo de efectivo para determinar las necesidades de efectivo para cada sucursal.

In [1]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.exceptions import NotFittedError
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from keras.metrics import mean_squared_error as mse 
from tensorflow.keras.losses import MeanSquaredError
from datetime import timedelta, datetime
from pandas.tseries.offsets import CustomBusinessDay
import pandas as pd
import numpy as np
import Funciones as f
import pickle
import os
import joblib

## 4.1 Cargamos la base de datos
### Cargamos la base de datos con las características temporales creadas en el 3. Entrenamiento de Modelos

In [2]:
# Ruta donde se guardó el archivo de los días festivos
ruta_archivo_flujo_efectivo_estado = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/df_mod.pkl"

with open(ruta_archivo_flujo_efectivo_estado, "rb") as archivo:
    df_mod = pickle.load(archivo)

print("Base de datos cargada correctamente.")

Base de datos cargada correctamente.


In [ ]:
# Ejemplo para el estado seleccionado:
estado_seleccionado = "guerrero"  # Cambia esto por el estado que desees filtrar
flujo_estado_características = f.filtrar_estado(df_mod, estado_seleccionado)

# Mostrar las primeras filas del resultado
flujo_estado_características.head(10)

In [ ]:
flujo_estado_características.dtypes

## 4.2 Cargamos los modelos y escaladores
### Cargamos el modelo entrenado en cuaderno '3. Entrenamiento de Modelos' que haya resultado con el menor error y los escaladores entrenados para cada sucursal.

In [ ]:
# Ruta base del proyecto
ruta_base = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV"
# Ruta al archivo pickle con los mejores modelos por sucursal
ruta_pickle_modelos = os.path.join(ruta_base, "modelos", "mejores", "mejores_modelos.pkl")

# Cargar diccionario
try:
    mejores_modelos = joblib.load(ruta_pickle_modelos)
    print(f"✅ Diccionario 'mejores_modelos' cargado. Total sucursales: {len(mejores_modelos)}")
except Exception as e:
    print(f"❌ Error al cargar 'mejores_modelos': {e}")
    mejores_modelos = {}

In [ ]:
# Diccionario para almacenar los modelos cargados
modelos_cargados = {}

for id_sucursal, mejor_modelo in mejores_modelos.items():
    modelo_lower = mejor_modelo.lower()

    if mejor_modelo == "XGBoost":
        ruta_modelo = os.path.join(ruta_base, "modelos", "mejores", modelo_lower, f"sucursal_{id_sucursal}_mejor_modelo_xgb.pkl")
        try:
            modelos_cargados[id_sucursal] = joblib.load(ruta_modelo)
            print(f"✅ Modelo XGBoost cargado para sucursal {id_sucursal}")
        except Exception as e:
            print(f"❌ Error al cargar modelo XGBoost para sucursal {id_sucursal}: {e}")

    elif mejor_modelo == "LSTM":
        ruta_modelo = os.path.join(ruta_base, "modelos", "mejores", modelo_lower, f"sucursal_{id_sucursal}_mejor_modelo_lstm.h5")
        try:
            modelo_lstm = load_model(ruta_modelo, compile=False)
            modelo_lstm.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError()) 
            modelos_cargados[id_sucursal] = modelo_lstm
            print(f"✅ Modelo LSTM cargado para sucursal {id_sucursal}")
        except Exception as e:
            print(f"❌ Error al cargar modelo LSTM para sucursal {id_sucursal}: {e}")

In [ ]:
# Ruta base del proyecto
ruta_base = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV"

# Diccionarios donde se cargarán los escaladores
scalers_X_cargados = {}
scalers_y_cargados = {}

# Diccionario con el mejor modelo por sucursal (puede venir de un archivo .json, .pkl o ya estar definido)
# Ejemplo de estructura esperada:
# mejores_modelos = {'001': 'XGBoost', '002': 'LSTM', ...}

for id_sucursal, mejor_modelo in mejores_modelos.items():
    modelo_lower = mejor_modelo.lower()

    # Rutas completas
    ruta_scaler_X = os.path.join(ruta_base, "escaladores", modelo_lower, "features", f"sucursal_{id_sucursal}_scaler_X.pkl")
    ruta_scaler_y = os.path.join(ruta_base, "escaladores", modelo_lower, "targets", f"sucursal_{id_sucursal}_scaler_y.pkl")

    try:
        scalers_X_cargados[id_sucursal] = joblib.load(ruta_scaler_X)
        scalers_y_cargados[id_sucursal] = joblib.load(ruta_scaler_y)
        print(f"✅ Escaladores cargados para sucursal {id_sucursal} ({mejor_modelo})")
    except Exception as e:
        print(f"❌ Error al cargar escaladores para sucursal {id_sucursal}: {e}")

## 4.3 Entrenamos los modelos auxiliares
### Se entrenan modelos (auxiliares) para las entradas y salidas que se utilizarán para pronosticar el flujo de efectivo de las sucursales. 

In [ ]:
# Nuevo parámetro
window_size = 7

# Nuevas columnas derivadas
def generar_features_temporales(df, col):
    df[f'lag_1_{col}'] = df[col].shift(1)
    df[f'lag_2_{col}'] = df[col].shift(2)
    df[f'media_movil_3_{col}'] = df[col].rolling(3).mean()
    df[f'media_movil_5_{col}'] = df[col].rolling(5).mean()
    return df

# Features base de calendario
features_base = ['dia_semana', 'semana_año', 'mes', 'año']

# Obtener lista única de sucursales
id_sucursales = flujo_estado_características["id_sucursal"].astype(str).unique()

# Entrenamiento de modelos auxiliares
for id_sucursal in id_sucursales:
    try:
        if id_sucursal not in mejores_modelos:
            print(f"⚠️  No se encontró modelo para sucursal {id_sucursal}, se omite.")
            continue

        mejor_modelo = mejores_modelos[id_sucursal]
        print(f"➡ Entrenando modelos auxiliares para sucursal {id_sucursal} ({mejor_modelo})")

        df_sucursal = flujo_estado_características[
            flujo_estado_características["id_sucursal"].astype(str) == id_sucursal
        ].copy()

        # Generar nuevas features
        df_sucursal = generar_features_temporales(df_sucursal, 'entradas')
        df_sucursal = generar_features_temporales(df_sucursal, 'salidas')

        # Eliminar filas con NaN generadas por lags y medias móviles
        df_sucursal.dropna(inplace=True)

        features_aux = features_base + [
            'lag_1_entradas', 'lag_2_entradas', 'media_movil_3_entradas', 'media_movil_5_entradas',
            'lag_1_salidas', 'lag_2_salidas', 'media_movil_3_salidas', 'media_movil_5_salidas'
        ]

        scaler_X = MinMaxScaler()
        scaler_y_entradas = MinMaxScaler()
        scaler_y_salidas = MinMaxScaler()

        X_aux = scaler_X.fit_transform(df_sucursal[features_aux])
        y_entradas = scaler_y_entradas.fit_transform(df_sucursal[['entradas']])
        y_salidas = scaler_y_salidas.fit_transform(df_sucursal[['salidas']])

        modelo_lower = mejor_modelo.lower()

        # Guardar escaladores
        rutas_scalers = {
            "features": os.path.join(ruta_base, "escaladores", "auxiliares", modelo_lower, "features"),
            "entradas": os.path.join(ruta_base, "escaladores", "auxiliares", modelo_lower, "entradas"),
            "salidas": os.path.join(ruta_base, "escaladores", "auxiliares", modelo_lower, "salidas")
        }
        for path in rutas_scalers.values():
            os.makedirs(path, exist_ok=True)

        joblib.dump(scaler_X, os.path.join(rutas_scalers["features"], f"sucursal_{id_sucursal}_scaler_X.pkl"))
        joblib.dump(scaler_y_entradas, os.path.join(rutas_scalers["entradas"], f"sucursal_{id_sucursal}_scaler_y.pkl"))
        joblib.dump(scaler_y_salidas, os.path.join(rutas_scalers["salidas"], f"sucursal_{id_sucursal}_scaler_y.pkl"))

        ruta_modelos_aux = os.path.join(ruta_base, "modelos", "auxiliares", modelo_lower)
        os.makedirs(os.path.join(ruta_modelos_aux, "entradas"), exist_ok=True)
        os.makedirs(os.path.join(ruta_modelos_aux, "salidas"), exist_ok=True)

        if mejor_modelo == "XGBoost":
            modelo_entradas = XGBRegressor(objective='reg:squarederror', n_estimators=200, learning_rate=0.05, max_depth=6)
            modelo_salidas = XGBRegressor(objective='reg:squarederror', n_estimators=200, learning_rate=0.05, max_depth=6)

            modelo_entradas.fit(X_aux, y_entradas.ravel())
            modelo_salidas.fit(X_aux, y_salidas.ravel())

            joblib.dump(modelo_entradas, os.path.join(ruta_modelos_aux, "entradas", f"sucursal_{id_sucursal}_aux_entradas_xgb.pkl"))
            joblib.dump(modelo_salidas, os.path.join(ruta_modelos_aux, "salidas", f"sucursal_{id_sucursal}_aux_salidas_xgb.pkl"))

        else:
            def crear_lstm(input_shape):
                model = Sequential()
                model.add(LSTM(50, activation='relu', input_shape=input_shape))
                model.add(Dense(1))
                model.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError())
                return model

            # Reestructurar en ventanas temporales
            X_seq, y_ent_seq, y_sal_seq = [], [], []
            for i in range(window_size, len(X_aux)):
                X_seq.append(X_aux[i - window_size:i])
                y_ent_seq.append(y_entradas[i])
                y_sal_seq.append(y_salidas[i])

            X_seq = np.array(X_seq)
            y_ent_seq = np.array(y_ent_seq)
            y_sal_seq = np.array(y_sal_seq)

            modelo_entradas = crear_lstm((window_size, X_aux.shape[1]))
            modelo_salidas = crear_lstm((window_size, X_aux.shape[1]))

            modelo_entradas.fit(X_seq, y_ent_seq, epochs=50, verbose=0)
            modelo_salidas.fit(X_seq, y_sal_seq, epochs=50, verbose=0)

            modelo_entradas.save(
                os.path.join(ruta_modelos_aux, "entradas", f"sucursal_{id_sucursal}_aux_entradas_lstm.h5"),
                include_optimizer=False
            )
            modelo_salidas.save(
                os.path.join(ruta_modelos_aux, "salidas", f"sucursal_{id_sucursal}_aux_salidas_lstm.h5"),
                include_optimizer=False
            )

        print(f"✅ Modelos auxiliares guardados para sucursal {id_sucursal}\n")

    except Exception as e:
        print(f"❌ Error en sucursal {id_sucursal}: {e}\n")

## 4.4 Pronósticos de las Necesidades de Efectivo
### Hacemos los pronósticos fuera de la muestra de datos 'out of sample' con el modelo que haya resultado con el menor error 

In [ ]:
# Asegurar que la columna 'fecha' está en formato datetime
flujo_estado_características['fecha'] = pd.to_datetime(flujo_estado_características['fecha'])

# Asegurar que id_sucursal sea texto y sin espacios
flujo_estado_características['id_sucursal'] = flujo_estado_características['id_sucursal'].astype(str).str.strip()

# Días de la semana (0=Lunes, 6=Domingo)
dias_semana_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# --------------------------------------------------------------------
# 1. Crear diccionario con fechas laborales por sucursal
# --------------------------------------------------------------------
fechas_laborales = {}
dias_operacion_por_sucursal = {}

for id_sucursal, df_sucursal in flujo_estado_características.groupby('id_sucursal'):
    # Si no hay fechas válidas, omitir
    if df_sucursal['fecha'].isna().all():
        print(f"⚠️ Sucursal {id_sucursal} sin fechas válidas. Se omite.")
        continue

    # Determinar días de operación
    dias_operativos = sorted(df_sucursal['fecha'].dt.dayofweek.dropna().unique())
    if not dias_operativos:
        print(f"⚠️ Sucursal {id_sucursal} sin días operativos. Se omite.")
        continue
    dias_operacion_por_sucursal[id_sucursal] = dias_operativos

    # Convertir a formato weekmask
    weekmask = ' '.join([dias_semana_labels[d] for d in dias_operativos])
    frecuencia = CustomBusinessDay(weekmask=weekmask)

    # Calcular fechas mín y máx
    fecha_min = df_sucursal['fecha'].min()
    fecha_max = df_sucursal['fecha'].max()

    # Validar NaT y corregir
    if pd.isna(fecha_min):
        print(f"⚠️ fecha_min inválida para sucursal {id_sucursal}, se ajusta a hoy.")
        fecha_min = pd.Timestamp.today()

    if pd.isna(fecha_max):
        print(f"⚠️ fecha_max inválida para sucursal {id_sucursal}, se ajusta a hoy + 60 días.")
        fecha_max = pd.Timestamp.today()

    # Extender el rango de fecha para futuras predicciones
    fecha_max_ext = fecha_max + pd.Timedelta(days=60)

    # Generar fechas laborales
    try:
        fechas = pd.date_range(start=fecha_min, end=fecha_max_ext, freq=frecuencia)
        if len(fechas) == 0:
            print(f"⚠️ Calendario vacío para sucursal {id_sucursal}, se omite.")
            continue
        fechas_laborales[id_sucursal] = fechas
    except Exception as e:
        print(f"❌ Error generando fechas para sucursal {id_sucursal}: {e}")

# --------------------------------------------------------------------
# 2. Crear diccionario de entrenamiento por sucursal
# --------------------------------------------------------------------
entrenamientos = {
    id_sucursal: df_sucursal.copy()
    for id_sucursal, df_sucursal in flujo_estado_características.groupby('id_sucursal')
}

# --------------------------------------------------------------------
# 3. Filtrar solo los días laborales por sucursal
# --------------------------------------------------------------------
entrenamientos_filtrados = {}

for id_sucursal, df in entrenamientos.items():
    if id_sucursal in fechas_laborales:
        fechas_validas = fechas_laborales[id_sucursal]
        df_filtrado = df[df['fecha'].isin(fechas_validas)].copy()
        entrenamientos_filtrados[id_sucursal] = df_filtrado

        # Verificación opcional
        print(f"✅ Sucursal {id_sucursal} - Días laborales: {df_filtrado['fecha'].nunique()}")
    else:
        print(f"⚠️ No se encontraron fechas laborales para sucursal {id_sucursal}")

In [10]:
def cargar_modelos_auxiliares_por_sucursal(id_sucursal, mejores_modelos, ruta_base):
    if id_sucursal not in mejores_modelos:
        print(f"⚠️  Sucursal {id_sucursal} no tiene modelo principal ganador registrado.")
        return None, None, None, None, None

    tipo_modelo = mejores_modelos[id_sucursal].lower()  # 'xgboost' o 'lstm'

    # Rutas base para auxiliares
    ruta_modelos_aux = os.path.join(ruta_base, "modelos", "auxiliares", tipo_modelo)
    ruta_scalers_aux = os.path.join(ruta_base, "escaladores", "auxiliares", tipo_modelo)

    try:
        if tipo_modelo == "xgboost":
            modelo_entradas = joblib.load(os.path.join(ruta_modelos_aux, "entradas", f"sucursal_{id_sucursal}_modelo_aux_entradas.pkl"))
            modelo_salidas = joblib.load(os.path.join(ruta_modelos_aux, "salidas", f"sucursal_{id_sucursal}_modelo_aux_salidas.pkl"))

        elif tipo_modelo == "lstm":
            modelo_entradas = load_model(
                os.path.join(ruta_modelos_aux, "entradas", f"sucursal_{id_sucursal}_modelo_aux_entradas.h5"),
                compile=False
            )
            modelo_entradas.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError())

            modelo_salidas = load_model(
                os.path.join(ruta_modelos_aux, "salidas", f"sucursal_{id_sucursal}_modelo_aux_salidas.h5"),
                compile=False
            )
            modelo_salidas.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError())

        else:
            print(f"❌ Tipo de modelo no reconocido para sucursal {id_sucursal}: {tipo_modelo}")
            return None, None, None, None, None

        # Cargar escaladores
        scaler_X = joblib.load(os.path.join(ruta_scalers_aux, "features", f"sucursal_{id_sucursal}_scaler_X.pkl"))
        scaler_y_entradas = joblib.load(os.path.join(ruta_scalers_aux, "entradas", f"sucursal_{id_sucursal}_scaler_y_entradas.pkl"))
        scaler_y_salidas = joblib.load(os.path.join(ruta_scalers_aux, "salidas", f"sucursal_{id_sucursal}_scaler_y_salidas.pkl"))

        print(f"✅ Modelos auxiliares cargados para sucursal {id_sucursal}")
        return modelo_entradas, modelo_salidas, scaler_X, scaler_y_entradas, scaler_y_salidas

    except Exception as e:
        print(f"❌ Error al cargar modelos auxiliares para sucursal {id_sucursal}: {e}")
        return None, None, None, None, None


In [ ]:
print(df['id_sucursal'].unique())
print(df.groupby('id_sucursal')['fecha'].min())
print(df.groupby('id_sucursal')['fecha'].max())


In [ ]:
# Etiquetas de días
dias_semana_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Lista de sucursales objetivo
sucursales_objetivo = flujo_estado_características["id_sucursal"].unique()

# Diccionario para almacenar predicciones por sucursal
predicciones_por_sucursal = {}

# Número de días a predecir
n_dias = 12

# Validar que todas las sucursales tengan días operativos asignados (default Lunes-Viernes)
for suc in sucursales_objetivo:
    if suc not in dias_operacion_por_sucursal or not dias_operacion_por_sucursal[suc]:
        dias_operacion_por_sucursal[suc] = ["Mon", "Tue", "Wed", "Thu", "Fri"]

print("🔁 Iniciando loop de predicción autoregresiva...\n")

for id_sucursal in sucursales_objetivo:
    print(f"\n🔁 Ejecutando predicción autoregresiva para sucursal {id_sucursal}")

    try:
        # Filtrar datos históricos de la sucursal
        df_sucursal = df[df['id_sucursal'] == id_sucursal].copy()
        df_sucursal = df_sucursal.sort_values('fecha')
        
        # Validar que haya datos y fechas válidas
        if df_sucursal.empty or df_sucursal['fecha'].isna().all():
            print(f"⚠️ Sucursal {id_sucursal} sin datos históricos válidos. Se omite.")
            continue
        
        # Fecha máxima histórica
        fecha_max = df_sucursal['fecha'].max()
        if pd.isna(fecha_max):
            print(f"⚠️ Sucursal {id_sucursal} con fecha máxima inválida. Se omite.")
            continue

        # Obtener días operativos de la sucursal y armar weekmask
        dias_operativos = dias_operacion_por_sucursal[id_sucursal]
        
        # Convertir días numéricos a etiquetas si no son strings
        if all(isinstance(d, (int, np.integer)) for d in dias_operativos):
            dias_operativos = [dias_semana_labels[d] for d in dias_operativos]
        
        weekmask = " ".join(dias_operativos)

        # Crear frecuencia personalizada con CustomBusinessDay
        dias_laborales = CustomBusinessDay(weekmask=weekmask)

        # Generar fechas futuras laborales personalizadas
        fechas_futuras = pd.date_range(
            start=fecha_max + pd.Timedelta(days=1),
            periods=n_dias,
            freq=dias_laborales)

        # Inicializar DataFrame para predicciones futuras
        df_futuro = pd.DataFrame(index=fechas_futuras)
        df_futuro.index.name = "fecha"
        df_futuro["id_sucursal"] = id_sucursal

        # Cargar modelos y escaladores auxiliares
        modelo_entradas, modelo_salidas, scaler_X_aux, scaler_y_entradas, scaler_y_salidas = cargar_modelos_auxiliares_por_sucursal(
            id_sucursal, mejores_modelos, ruta_base
        )

        if None in [modelo_entradas, modelo_salidas, scaler_X_aux, scaler_y_entradas, scaler_y_salidas]:
            print(f"⚠️  Saltando sucursal {id_sucursal} por error en carga de modelos.")
            continue

        # Obtener modelo principal y escaladores para la sucursal
        tipo_modelo = mejores_modelos[id_sucursal].lower()
        modelo = modelos_cargados[tipo_modelo][id_sucursal]
        scaler_X = scalers_X_cargados[tipo_modelo][id_sucursal]
        scaler_y = scalers_y_cargados[tipo_modelo][id_sucursal]

        # Iterar autoregresivamente por cada fecha futura
        for fecha in fechas_futuras:
            # Concatenar datos históricos + predicciones previas
            df_temp = pd.concat([df_sucursal, df_futuro[df_futuro.index < fecha]])
            df_temp = generar_variables_temporales(df_temp)

            # Obtener la fila con la fecha actual
            fila_actual = df_temp.loc[[fecha]].copy()

            # Seleccionar features auxiliares
            features_aux = ["media_movil_5", "media_movil_10", "lag_1", "lag_2", "lag_3", "lag_5"]
            X_aux = fila_actual[features_aux].values

            # Escalar features auxiliares
            X_aux_scaled = scaler_X_aux.transform(X_aux)

            # Predecir entradas y salidas
            if tipo_modelo == "xgboost":
                pred_entradas = modelo_entradas.predict(X_aux_scaled)
                pred_salidas = modelo_salidas.predict(X_aux_scaled)
            else:
                # Reshape para LSTM [samples, timesteps, features]
                X_aux_scaled_lstm = X_aux_scaled.reshape((1, X_aux_scaled.shape[0], X_aux_scaled.shape[1]))
                pred_entradas = modelo_entradas.predict(X_aux_scaled_lstm)
                pred_salidas = modelo_salidas.predict(X_aux_scaled_lstm)

                # Convertir a escalares si es necesario
                pred_entradas = np.array(pred_entradas).squeeze()
                pred_salidas = np.array(pred_salidas).squeeze()

            # Invertir escalado
            pred_entradas = np.ravel(scaler_y_entradas.inverse_transform(np.array(pred_entradas).reshape(-1, 1)))[0]
            pred_salidas = np.ravel(scaler_y_salidas.inverse_transform(np.array(pred_salidas).reshape(-1, 1)))[0]

            # Insertar predicciones al DataFrame futuro
            df_futuro.loc[fecha, "entradas"] = pred_entradas
            df_futuro.loc[fecha, "salidas"] = pred_salidas

            # Recalcular variables temporales para predecir flujo_efectivo
            df_temp = pd.concat([df_sucursal, df_futuro[df_futuro.index <= fecha]])
            df_temp = generar_variables_temporales(df_temp)

            # Extraer features del modelo principal
            fila_actual = df_temp.loc[[fecha]].copy()
            features = ['entradas', 'salidas', 'no_laborales', 'festivo_mx',
                        'dia_semana', 'semana_año', 'mes', 'año',
                        'media_movil_3', 'media_movil_5', 'media_movil_10', 'media_movil_14',
                        'lag_1', 'lag_2', 'lag_3', 'lag_5']
            X_flujo = fila_actual[features].values
            X_flujo_scaled = scaler_X.transform(X_flujo)

            # Predecir flujo_efectivo
            if tipo_modelo == "xgboost":
                pred_flujo = modelo.predict(X_flujo_scaled)
            else:
                X_flujo_scaled = X_flujo_scaled.reshape((1, X_flujo_scaled.shape[0], X_flujo_scaled.shape[1]))
                pred_flujo = modelo.predict(X_flujo_scaled)
                pred_flujo = np.array(pred_flujo).squeeze()

            pred_flujo = np.ravel(scaler_y.inverse_transform(np.array(pred_flujo).reshape(-1, 1)))[0]
            df_futuro.loc[fecha, "flujo_efectivo"] = pred_flujo

        # Guardar resultados para esta sucursal
        print(f"✅ Predicciones generadas para sucursal {id_sucursal}")
        # df_futuro ahora contiene entradas, salidas y flujo_efectivo predichos

    except Exception as e:
        print(f"❌ Error en sucursal {id_sucursal}: {e}")

print("\n✅ Loop completado.")

In [ ]:
# Concatenar todas las predicciones en un único DataFrame
df_predicciones_final = pd.concat(predicciones_por_sucursal.values(), ignore_index=True)

# Rutas de guardado
ruta_csv = os.path.join(ruta_base, "predicciones", "predicciones_futuras.csv")
ruta_pkl = os.path.join(ruta_base, "predicciones", "predicciones_futuras.pkl")

# Crear carpeta si no existe
os.makedirs(os.path.dirname(ruta_csv), exist_ok=True)

# Guardar como CSV
df_predicciones_final.to_csv(ruta_csv, index=False)
print(f"✅ Predicciones guardadas en CSV: {ruta_csv}")

# Guardar como Pickle
df_predicciones_final.to_pickle(ruta_pkl)
print(f"✅ Predicciones guardadas en Pickle: {ruta_pkl}")
